# Retail Sales Analytics – Data Cleaning & ETL
---
**Project:** Capstone Data Visualization & Analytics  
**Goal:** Transform raw data into a clean, analysis-ready format and export to the processed data directory.

## 1. Setup & Data Loading
Importing necessary libraries and loading the raw dataset. Supports both local execution and Google Colab with encoding fallback.

In [10]:
import pandas as pd
import numpy as np
import os
import sys

raw_file_path = None

# 1. Handle Google Colab Environment
if 'google.colab' in sys.modules:
    print("Detected Google Colab Environment.")
    raw_file_path = '/content/superstore.csv'
    
    if not os.path.exists(raw_file_path):
        print("\n--- FILE NOT FOUND ---")
        print("Please upload the 'superstore.csv' file from your local computer.")
        from google.colab import files
        uploaded = files.upload()
        
        if 'superstore.csv' not in uploaded and len(uploaded) > 0:
            uploaded_filename = list(uploaded.keys())[0]
            os.rename(uploaded_filename, 'superstore.csv')
            print(f"Renamed uploaded file '{uploaded_filename}' to 'superstore.csv'")

# 2. Handle Local Environment
else:
    potential_paths = [
        'data/raw/superstore.csv',
        '../data/raw/superstore.csv',
        '../../data/raw/superstore.csv',
        './superstore.csv',
        '/Users/kritgarg/Desktop/Hopper_Horizon_RetailSalesAnalytics/data/raw/superstore.csv'
    ]
    for path in potential_paths:
        if os.path.exists(path):
            raw_file_path = path
            print(f"Found local file at: {path}")
            break

# 3. Load Data with Encoding Fallback
if raw_file_path and os.path.exists(raw_file_path):
    try:
        # Try default UTF-8 first
        df = pd.read_csv(raw_file_path)
        print("Loaded with UTF-8 encoding.")
    except UnicodeDecodeError:
        try:
            # Try Latin-1 (Common for Superstore datasets)
            df = pd.read_csv(raw_file_path, encoding='ISO-8859-1')
            print("Loaded with ISO-8859-1 encoding.")
        except UnicodeDecodeError:
            # Try Windows-1252 as a final common fallback
            df = pd.read_csv(raw_file_path, encoding='Windows-1252')
            print("Loaded with Windows-1252 encoding.")
            
    print(f"\nDataset loaded successfully! Shape: {df.shape}")
    display(df.head())
else:
    raise FileNotFoundError("Could not locate 'superstore.csv'.")

Found local file at: ../data/raw/superstore.csv
Loaded with ISO-8859-1 encoding.

Dataset loaded successfully! Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 2. Handle Missing Values
Checking for null values and applying logical handling (imputation or removal).

In [11]:
# Detect null values
null_counts = df.isnull().sum()
print("Null counts per column:")
print(null_counts[null_counts > 0])

# Handling missing values (if any)
if df.isnull().values.any():
    # Filling missing Postal Codes with a placeholder
    if 'Postal Code' in df.columns:
        df['Postal Code'] = df['Postal Code'].fillna(0).astype(int)

print(f"Shape after handling nulls: {df.shape}")

Null counts per column:
Series([], dtype: int64)
Shape after handling nulls: (9994, 21)


## 3. Remove Duplicate Rows
Ensuring each transaction is unique.

In [12]:
initial_count = len(df)
df.drop_duplicates(inplace=True)
removed_count = initial_count - len(df)

print(f"Removed {removed_count} duplicate rows.")
print(f"Current shape: {df.shape}")

Removed 0 duplicate rows.
Current shape: (9994, 21)


## 4. Fix Date Columns
Converting `Order Date` and `Ship Date` to proper datetime objects for time-series analysis.

In [13]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

print("Date columns converted successfully.")
df[['Order Date', 'Ship Date']].dtypes

Date columns converted successfully.


Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object

## 5. Standardize Text Columns
Cleaning strings to avoid duplicates due to case sensitivity or leading/trailing spaces.

In [14]:
text_columns = ['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Region', 'Category', 'Sub-Category']

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()  # Remove leading/trailing spaces
        df[col] = df[col].str.title()              # Standardize to Title Case

print("Text columns standardized.")

Text columns standardized.


## 6. Validate Numeric Columns
Ensuring Sales, Quantity, Discount, and Profit are in the correct format and range.

In [15]:
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Handle any potential conversion errors (NaNs from coerce)
df.dropna(subset=[col for col in numeric_cols if col in df.columns], inplace=True)

print("Numeric columns validated.")
df[[col for col in numeric_cols if col in df.columns]].describe()

Numeric columns validated.


,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,3.789574,0.156203,28.656896
std,623.245101,2.225110,0.206452,234.260108
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.280000,2.000000,0.000000,1.728750
50%,54.490000,3.000000,0.200000,8.666500
75%,209.940000,5.000000,0.200000,29.364000
max,22638.480000,14.000000,0.800000,8399.976000


## 7. Feature Engineering
Creating new columns that will be useful for visualization and analytics.

In [16]:
# Extracting Year and Month from Order Date
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month_name()

# Calculating Profit Margin
# (Profit / Sales) * 100
if 'Profit' in df.columns and 'Sales' in df.columns:
    df['Profit Margin'] = (df['Profit'] / df['Sales']) * 100

print("New features created: Order Year, Order Month, Profit Margin")
df[['Order Date', 'Order Year', 'Order Month', 'Profit Margin']].head()

New features created: Order Year, Order Month, Profit Margin


,Order Date,Order Year,Order Month,Profit Margin
0,2016-11-08,2016,November,16.00
1,2016-11-08,2016,November,30.00
2,2016-06-12,2016,June,47.00
3,2015-10-11,2015,October,-40.00
4,2015-10-11,2015,October,11.25


## 8. Final Quality Check
Confirming data integrity before export.

In [17]:
print(f"Final Dataset Shape: {df.shape}")
print("\nSummary of nulls:")
print(df.isnull().sum().sum())

df.info()

Final Dataset Shape: (9994, 24)

Summary of nulls:
0
<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 24 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   str           
 2   Order Date     9994 non-null   datetime64[us]
 3   Ship Date      9994 non-null   datetime64[us]
 4   Ship Mode      9994 non-null   str           
 5   Customer ID    9994 non-null   str           
 6   Customer Name  9994 non-null   str           
 7   Segment        9994 non-null   str           
 8   Country        9994 non-null   str           
 9   City           9994 non-null   str           
 10  State          9994 non-null   str           
 11  Postal Code    9994 non-null   int64         
 12  Region         9994 non-null   str           
 13  Product ID     9994 non-null   str           
 14  Category       9994 non-null  

## 9. Export Cleaned Data
Saving the transformed dataset to the processed folder.

In [18]:
if 'google.colab' in sys.modules:
    # In Colab, just save to /content and trigger download
    output_path = '/content/cleaned_sales.csv'
    df.to_csv(output_path, index=False)
    print(f"Cleaned data exported to Colab environment at: {output_path}")
    
    print("Triggering download...")
    from google.colab import files
    files.download(output_path)
    
else:
    output_dir = 'data/processed/'
    if not os.path.exists(output_dir):
        if os.path.exists('../data'):
            output_dir = '../data/processed/'
        else:
            output_dir = './' 
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    output_path = os.path.join(output_dir, 'cleaned_sales.csv')
    df.to_csv(output_path, index=False)
    print(f"Cleaned data exported locally to: {output_path}")

Cleaned data exported locally to: ../data/processed/cleaned_sales.csv


## Summary of Transformations

| Transformation | Action Taken |
| :--- | :--- |
| **Missing Values** | Checked for nulls and handled Postal Code placeholders. |
| **Duplicates** | Removed all redundant rows. |
| **Date Casting** | Converted `Order Date` and `Ship Date` to datetime objects. |
| **Text Standardization** | Trimmed whitespace and applied Title Case to categorical columns. |
| **Numeric Validation** | Coerced Sales, Quantity, Discount, and Profit to numeric types. |
| **Feature Engineering** | Added `Order Year`, `Order Month`, and `Profit Margin`. |
| **Export** | Saved the final clean dataset to `cleaned_sales.csv`. |